# Laboratorium 3: Regresja, modele zespołowe, strojenie, interpretacja

Zbiór danych: Ames Housing (OpenML, `house_prices`), cel: cena sprzedaży domu `SalePrice`.
Środowisko: Google Colab, CPU wystarczy.

Numeracja nagłówków odpowiada numeracji zadań i poleceń w instrukcji (np. 2.3 to polecenie 3 z zadania 2).

## Przygotowanie

In [ ]:
# W tym laboratorium wszystkie biblioteki są domyślnie dostępne w Colabie, więc nic nie trzeba instalować.
# Wymagany scikit-learn 1.4 lub nowszy.
import sklearn
print(sklearn.__version__)

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import loguniform, randint

from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.datasets import fetch_openml
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import (mean_absolute_error, r2_score, root_mean_squared_error,
                             root_mean_squared_log_error)
from sklearn.model_selection import (GridSearchCV, KFold, RandomizedSearchCV, cross_val_predict,
                                     cross_validate, train_test_split, validation_curve)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

RANDOM_STATE = 42
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams["figure.dpi"] = 100

## Zadanie 1: Dane i zmienna docelowa

### 1.1 Wczytanie danych

In [ ]:
# Pobierz zbiór fetch_openml(name="house_prices", as_frame=True).
# Zapisz cechy do X (bez kolumny Id) i cel do y. Wypisz kształt X.

### 1.2 Podział na zbiór treningowy i testowy

In [ ]:
# Podziel dane 80/20 z random_state=RANDOM_STATE. Od tej chwili zbiór testowy jest odłożony.

### 1.3 Rozkład ceny

In [ ]:
# Opisz statystyki ceny w zbiorze treningowym (describe).
# Narysuj histogram ceny i histogram log1p(ceny) obok siebie, w tytułach podaj skośność (scipy.stats.skew).

### 1.4 Zależności z ceną

In [ ]:
# Policz korelację Spearmana cech liczbowych z log1p(ceny) (corrwith), wypisz 10 najsilniejszych co do modułu.
# Narysuj wykres rozrzutu GrLivArea względem ceny oraz OverallQual względem log1p(ceny).

### 1.5 Nietypowe obserwacje

In [ ]:
# Wypisz domy treningowe o GrLivArea > 4000 razem z ceną, jakością (OverallQual) i warunkami sprzedaży (SaleCondition).

## Zadanie 2: Przetwarzanie i baseline

### 2.1 Braki danych

In [ ]:
# Wypisz kolumny z brakami i liczbę braków (zbiór treningowy).
# Dla par (kolumna opisowa, kolumna liczbowa), np. (PoolQC, PoolArea), sprawdź,
# jak często NaN w pierwszej zbiega się z zerem w drugiej.

### 2.2 Opis kolumn

In [ ]:
# Kolejność kategorii porządkowych, przepisana z opisu zbioru (data_description.txt).
# Etykieta "brak" oznacza brak danego elementu w domu i jest najniższym poziomem.
QUAL = ["brak", "Po", "Fa", "TA", "Gd", "Ex"]
ORD = {
    **{c: QUAL for c in ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC",
                         "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "PoolQC"]},
    "BsmtExposure": ["brak", "No", "Mn", "Av", "Gd"],
    "BsmtFinType1": ["brak", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "BsmtFinType2": ["brak", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "GarageFinish": ["brak", "Unf", "RFn", "Fin"],
    "LotShape": ["IR3", "IR2", "IR1", "Reg"],
    "LandSlope": ["Sev", "Mod", "Gtl"],
    "Functional": ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],
    "PavedDrive": ["N", "P", "Y"],
    "CentralAir": ["N", "Y"],
    "Utilities": ["NoSeWa", "AllPub"],
}

# Podziel kolumny na grupy: porządkowe (ord_cols), nominalne (nom_cols), liczbowe (num_cols).
# Nominalne dziel dalej: z NA oznaczającym brak elementu, z NA oznaczającym brak informacji, pozostałe.
# Liczbowe z brakami (num_na) trzymaj osobno od pozostałych (num_plain).

### 2.3 Przetwarzanie wstępne

In [ ]:
# Napisz make_preprocessor() zwracającą ColumnTransformer, a make_model(estimator) zwracającą
# TransformedTargetRegressor (log1p / expm1) opakowujący Pipeline(prep, model).
# Dopasuj preprocesor na X_train i sprawdź kształt wyniku oraz brak NaN.

### 2.4 Metryki i walidacja krzyżowa

In [ ]:
SCORING = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2",
    "RMSLE": "neg_root_mean_squared_log_error",
}
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = {}

# Napisz evaluate(name, model): cross_validate na (X_train, y_train) z SCORING,
# średnie metryki (z poprawnym znakiem) i średni czas dopasowania zapisz w results[name].
# Napisz summary(): tabela z results (pd.DataFrame(results).T) zaokrąglona do czytelnej postaci.

### 2.5 Baseline

In [ ]:
# Oceń: Dummy (średnia i mediana), Ridge bez transformacji celu, Ridge z logarytmem celu.
# Wypisz summary().

In [ ]:
# Policz, ile predykcji Ridge'a bez logarytmu (cross_val_predict) jest ujemnych.

## Zadanie 3: Modele liniowe i regularyzacja

### 3.1 Cztery modele z domyślnymi ustawieniami

In [ ]:
# Oceń LinearRegression, Ridge, Lasso i ElasticNet z domyślnymi parametrami (make_model + evaluate).

### 3.2 Siła regularyzacji w Ridge

In [ ]:
# Napisz funkcję liczącą krzywą walidacyjną (validation_curve) dla make_model(estimator) po parametrze alpha.
# W zagnieżdżonym modelu parametr nazywa się regressor__model__alpha.
# Dla Ridge użyj alpha z np.logspace(-1, 2.5, 12), metryka neg_root_mean_squared_log_error.
# Narysuj wynik treningowy i walidacyjny w skali logarytmicznej osi x, zapisz najlepsze alpha.

### 3.3 Lasso i ElasticNet

In [ ]:
# Zrób to samo dla Lasso i ElasticNet (alpha z np.logspace(-4, -1, 10), max_iter=20000).
# Dopisz do tabeli wyników trzy modele z najlepszym alpha.

### 3.4 Ścieżki współczynników

In [ ]:
# Dopasuj preprocesor na X_train (prep), wyznacz macierz Z i nazwy cech (get_feature_names_out).
# Celem jest log1p(y_train). Policz ścieżkę dla Ridge i dla Lasso (pętla po alpha, w każdym kroku fit na Z).
# Narysuj trzy wykresy: współczynniki Ridge, współczynniki Lasso, liczba niezerowych współczynników Lasso.
# Dla najlepszego alpha Lasso podaj liczbę zerowych współczynników i 10 największych co do modułu.

### 3.5 Które domy psują RMSE

In [ ]:
# Wyznacz predykcje walidacyjne najlepszego Ridge'a (cross_val_predict) i wypisz 5 domów
# o największym błędzie bezwzględnym (cena, predykcja, GrLivArea, SaleCondition).
# Policz RMSE z tymi domami i bez dwóch największych błędów.

## Zadanie 4: Modele zespołowe

### 4.1 Las losowy, Extra Trees, gradient boosting

In [ ]:
# Oceń RandomForestRegressor (200 drzew), ten sam las z max_features=0.3, ExtraTreesRegressor (200 drzew)
# i HistGradientBoostingRegressor z domyślnymi parametrami. Pamiętaj o random_state.

### 4.2 Bagging i boosting: liczba drzew

In [ ]:
# Dla lasu losowego zmieniaj n_estimators (10, 30, 100, 200), dla HistGradientBoosting max_iter (25, 50, 100, 200).
# Użyj validation_curve na make_model(...) z metryką neg_root_mean_squared_log_error.
# Narysuj wynik treningowy i walidacyjny obok siebie, z tą samą osią y.

### 4.3 Rola logarytmu w modelach drzewiastych

In [ ]:
# Oceń HistGradientBoostingRegressor bez TransformedTargetRegressor (cena surowa) i porównaj z wersją z logarytmem.

### 4.4 Porównanie wszystkich modeli

In [ ]:
# Posortuj tabelę wyników po RMSLE i narysuj wykres słupkowy poziomy.

## Zadanie 5: Strojenie hiperparametrów

### 5.1 Przeszukiwanie siatki

In [ ]:
# Zbuduj HistGradientBoostingRegressor(max_iter=150) w make_model i przeszukaj siatkę GridSearchCV:
# learning_rate [0.05, 0.1, 0.2], max_leaf_nodes [6, 15], l2_regularization [0.0, 1.0].
# Mierz czas (time.perf_counter), wypisz liczbę kandydatów, najlepszy wynik i parametry.

### 5.2 Przeszukiwanie losowe z tym samym budżetem

In [ ]:
# Wykonaj RandomizedSearchCV z tą samą liczbą kandydatów (n_iter = rozmiar siatki).
# Rozkłady: learning_rate loguniform(0.02, 0.3), max_leaf_nodes randint(4, 32),
# min_samples_leaf randint(10, 60), l2_regularization loguniform(1e-3, 10).

### 5.3 Co zbadały obie metody

In [ ]:
# Narysuj dla obu metod wynik walidacyjny każdego kandydata względem learning_rate (skala log osi x)
# i zestaw w tabeli: liczba kandydatów, najlepszy RMSLE, czas.

### 5.4 Wybór modelu końcowego

In [ ]:
# Wybierz lepszy z dwóch modeli według wyniku walidacyjnego (best_estimator_ jest już dopasowany do całego X_train).
# Dopisz go do tabeli wyników (evaluate na clone(...)) i porównaj z Ridge strojonym i HistGradientBoosting domyślnym.

## Zadanie 6: Interpretacja i analiza błędów

### 6.1 Predykcje na zbiorze testowym

In [ ]:
# Model końcowy jest wybrany, więc po raz pierwszy używany jest zbiór testowy.
# Policz predykcje y_pred i wypisz MAE oraz RMSLE.

### 6.2 Ważność permutacyjna

In [ ]:
# permutation_importance na pełnym modelu (final_model), zbiór testowy, scoring neg_root_mean_squared_log_error,
# n_repeats=10. Narysuj 15 najważniejszych cech z odchyleniem standardowym.

### 6.3 Wykresy PDP i ICE

In [ ]:
# PartialDependenceDisplay.from_estimator dla GrLivArea, OverallQual i YearBuilt, kind="both", subsample=50.
# Wskazówka: kolumny całkowitoliczbowe trzeba wcześniej zamienić na float.

### 6.4 Wykresy reszt

In [ ]:
# Narysuj cenę rzeczywistą względem przewidzianej (z linią y = x) oraz błąd względny względem ceny przewidzianej.

### 6.5 Największe błędy

In [ ]:
# Zbierz w tabeli 8 domów z największym błędem względnym (Neighborhood, SaleCondition, OverallQual,
# GrLivArea, YearBuilt, cena, predykcja, błąd w procentach). Dodaj 3 domy o największym błędzie w dolarach.
# Policz medianę bezwzględnego błędu względnego w grupach SaleCondition.

## Zadanie 7: Ocena końcowa

### 7.1 Tabela wyników na zbiorze testowym

In [ ]:
# Dopasuj na X_train Dummy i Ridge z najlepszym alpha, weź final_model i policz MAE, RMSE, R2, RMSLE na zbiorze testowym.
# Dodaj kolumnę z RMSLE z walidacji krzyżowej (results) do porównania.

### 7.2 Niepewność oceny

In [ ]:
# Wyznacz 95-procentowy przedział bootstrapowy dla RMSLE na zbiorze testowym (2000 losowań ze zwracaniem
# kwadratów błędów logarytmicznych).

### 7.3 Wnioski

Napisz 4 do 6 zdań: który model wybierasz, jak wypada na teście względem walidacji, które cechy decydują o cenie i gdzie model się myli.